# Nemotron LoRA — train on Kaggle (free 2xT4, 4-bit QLoRA)

**Run HEADLESS:** Save Version -> **Save & Run All (Commit)**.

**Add Input:** **competition data** + model **`nemotron-3-nano-30b-a3b-bf16`** (publisher `metric`). **GPU T4 x2, Internet On.**

The model hard-requires mamba_ssm at import, so we pin **torch 2.7** (clearing Kaggle's PIP_CONSTRAINT) and install the matching prebuilt mamba wheels.

## 1. Code + torch 2.7 (force past Kaggle's pin) + deps
The assert fails loudly if torch 2.7 didn't take — so we never reach a mismatch.

In [ ]:
!git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
# Kaggle pins torch via PIP_CONSTRAINT; clear it so torch 2.7 actually installs
!PIP_CONSTRAINT= pip install -q torch==2.7.0
!pip install -q "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil einops
import torch
print("TORCH:", torch.__version__, "abi:", torch.compiled_with_cxx11_abi())
assert torch.__version__.startswith("2.7"), "torch is not 2.7 -> mamba wheels will mismatch"

## 2. Prebuilt mamba_ssm + causal_conv1d for torch 2.7 (~3 min)

In [ ]:
!pip install -q --no-deps 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1+cu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl' 'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'
!python -c "import causal_conv1d, mamba_ssm; print('mamba OK')"

## 3. Competition data (recursive find)

In [ ]:
import glob, os, shutil
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert hits, "train.csv not found — Add Input -> the competition"
shutil.copy(hits[0], 'data/train.csv'); print('train.csv <-', hits[0])

## 4. EDA + build the SFT data

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 5. Train (4-bit QLoRA on 2xT4)
Base from the attached model mount (no 60 GB download). 1 epoch + seq 768 for a first finish; bump later. Smoke test runs first.

In [ ]:
import os
os.environ['QUANT'] = '4bit'
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '14GiB'
os.environ['SFT_MAX_SEQ_LENGTH'] = '768'
os.environ['NUM_EPOCHS'] = '1'
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir /kaggle/working/lora_adapter

## 6. Package the submission

In [ ]:
!python scripts/05_package_submission.py --adapter-dir /kaggle/working/lora_adapter --output /kaggle/working/submission.zip